In [ ]:
import pandas as pd

CSV_URL = "https://data.cityofnewyork.us/api/views/h9gi-nx95/rows.csv"

df = pd.read_csv(CSV_URL, low_memory=False)
print("Loaded rows:", len(df))
df.head()

Loaded rows: 2244020


,CRASH DATE,CRASH TIME,BOROUGH,ZIP CODE,LATITUDE,LONGITUDE,LOCATION,ON STREET NAME,CROSS STREET NAME,OFF STREET NAME,...,CONTRIBUTING FACTOR VEHICLE 2,CONTRIBUTING FACTOR VEHICLE 3,CONTRIBUTING FACTOR VEHICLE 4,CONTRIBUTING FACTOR VEHICLE 5,COLLISION_ID,VEHICLE TYPE CODE 1,VEHICLE TYPE CODE 2,VEHICLE TYPE CODE 3,VEHICLE TYPE CODE 4,VEHICLE TYPE CODE 5
0,09/11/2021,2:39,NaN,NaN,NaN,NaN,NaN,WHITESTONE EXPRESSWAY,20 AVENUE,NaN,...,Unspecified,NaN,NaN,NaN,4455765,Sedan,Sedan,NaN,NaN,NaN
1,03/26/2022,11:45,NaN,NaN,NaN,NaN,NaN,QUEENSBORO BRIDGE UPPER,NaN,NaN,...,NaN,NaN,NaN,NaN,4513547,Sedan,NaN,NaN,NaN,NaN
2,11/01/2023,1:29,BROOKLYN,11230,40.62179,-73.970024,"(40.62179, -73.970024)",OCEAN PARKWAY,AVENUE K,NaN,...,Unspecified,Unspecified,NaN,NaN,4675373,Moped,Sedan,Sedan,NaN,NaN
3,06/29/2022,6:55,NaN,NaN,NaN,NaN,NaN,THROGS NECK BRIDGE,NaN,NaN,...,Unspecified,NaN,NaN,NaN,4541903,Sedan,Pick-up Truck,NaN,NaN,NaN
4,09/21/2022,13:21,NaN,NaN,NaN,NaN,NaN,BROOKLYN BRIDGE,NaN,NaN,...,Unspecified,NaN,NaN,NaN,4566131,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN,NaN


In [ ]:
def normalize_col(c):
    return (
        c.strip()
         .lower()
         .replace(" ", "_")
         .replace("/", "_")
         .replace("-", "_")
         .replace("__", "_")
    )

df.columns = [normalize_col(c) for c in df.columns]

# Common column name in NYC collisions: crash_date or crash_date/crash_time depending on export
# We'll handle both robustly.
if "crash_date" in df.columns and "crash_time" in df.columns:
    df["crash_datetime"] = pd.to_datetime(df["crash_date"].astype(str) + " " + df["crash_time"].astype(str), errors="coerce")
elif "crash_date" in df.columns:
    df["crash_datetime"] = pd.to_datetime(df["crash_date"], errors="coerce")
else:
    raise ValueError("Could not find crash_date in columns. Print df.columns and adjust parsing.")

df["crash_date"] = df["crash_datetime"].dt.date.astype("string")
df["crash_year"] = df["crash_datetime"].dt.year
df["crash_month"] = df["crash_datetime"].dt.month
df["crash_hour"] = df["crash_datetime"].dt.hour
df["day_of_week"] = df["crash_datetime"].dt.day_name()

# numeric safety
num_cols = [c for c in df.columns if c.startswith(("number_of_", "persons_", "pedestrians_", "cyclist", "motorist"))]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# latitude/longitude (if present)
for c in ["latitude", "longitude"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# intersection key (works even if one side is missing)
street_cols = [c for c in ["on_street_name", "cross_street_name", "off_street_name"] if c in df.columns]
for c in street_cols:
    df[c] = df[c].astype("string").str.strip().str.upper()

if "on_street_name" in df.columns and "cross_street_name" in df.columns:
    df["intersection"] = (df["on_street_name"].fillna("") + " @ " + df["cross_street_name"].fillna("")).str.replace(r"\s+", " ", regex=True)
else:
    df["intersection"] = None

# borough normalize
if "borough" in df.columns:
    df["borough"] = df["borough"].astype("string").str.strip().str.upper()

df = df.dropna(subset=["crash_datetime"])
print(df.shape)

(2244020, 35)


In [ ]:
vehicle_cols = [c for c in df.columns if c.startswith("vehicle_type_code")]
vehicle_cols[:10], len(vehicle_cols)

(['vehicle_type_code_1',
  'vehicle_type_code_2',
  'vehicle_type_code_3',
  'vehicle_type_code_4',
  'vehicle_type_code_5'],
 5)

In [ ]:
TRUCK_HINTS = [
    "TRUCK", "BOX TRUCK", "TRACTOR", "SEMI", "DELIVERY", "VAN", "AMAZON", "FEDEX", "UPS",
    "SPRY", "STEP VAN", "PANEL VAN", "COMMERCIAL"
]

def is_delivery_truck_row(row):
    vals = []
    for c in vehicle_cols:
        v = row.get(c)
        if pd.isna(v):
            continue
        vals.append(str(v).upper())
    text = " | ".join(vals)
    return any(h in text for h in TRUCK_HINTS)

df["has_delivery_truck"] = df.apply(is_delivery_truck_row, axis=1).astype(int)

In [ ]:
from sqlalchemy import create_engine, text

db_path = "/content/nyc_crashes.sqlite"
engine = create_engine(f"sqlite:///{db_path}")

# keep only columns that SQLite can handle reliably
# (SQLite is fine with strings, ints, floats, dates as strings)
df.to_sql("crashes", con=engine, if_exists="replace", index=False)

with engine.begin() as conn:
    conn.execute(text("CREATE INDEX IF NOT EXISTS idx_borough_date ON crashes(borough, crash_date)"))
    conn.execute(text("CREATE INDEX IF NOT EXISTS idx_dt ON crashes(crash_datetime)"))
    if "intersection" in df.columns:
        conn.execute(text("CREATE INDEX IF NOT EXISTS idx_intersection ON crashes(intersection)"))
    if "latitude" in df.columns and "longitude" in df.columns:
        conn.execute(text("CREATE INDEX IF NOT EXISTS idx_latlon ON crashes(latitude, longitude)"))

print("SQLite ready:", db_path)

SQLite ready: /content/nyc_crashes.sqlite


In [ ]:
ASSISTANT_NAME = "VisionZeroMate"
TAGLINE = "NYC Crash & Safety Assistant"

def one_liner(answer: str, why: str) -> str:
    """Return answer + exactly one short explanation sentence."""
    why = why.strip()
    if not why.endswith("."):
        why += "."
    return f"{answer}\n{why}"

In [ ]:
from getpass import getpass
import os

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


In [ ]:
!pip install langchain_groq
from langchain_groq import ChatGroq
import os

# You still need GROQ_API_KEY in env (you already did this part earlier)
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 13.6 MB/s eta 0:00:00


In [ ]:
from sqlalchemy import create_engine, text
import pandas as pd

DB_PATH = "/content/nyc_crashes.sqlite"
engine = create_engine(f"sqlite:///{DB_PATH}")

def run_sql(query: str, params: dict | None = None) -> pd.DataFrame:
    with engine.begin() as conn:
        return pd.read_sql(text(query), conn, params=params or {})

In [ ]:
run_sql("SELECT COUNT(*) AS total_rows FROM crashes")

,total_rows
0,2244020


In [ ]:
INTENT_SYSTEM = """
You are an intent router for a city crash analytics assistant.
Classify the user question into EXACTLY ONE intent:
- safety: hotspots, intersections, delivery trucks, cyclists, pedestrians, injuries, "where is worst"
- policy: trends over years/months, "last N years", impact, comparisons over time, Manhattan/Brooklyn trends
- risk: route or time-of-day risk, "is it safer at night", "risk for this route", "what time is riskiest"
Return ONLY JSON with keys: intent, entities.
entities can include: borough, mode (cyclist/pedestrian/motorist/persons), involves_delivery_truck (true/false),
time_window (e.g. "last 3 years"), hour_range (e.g. "16-19"), min_crashes (integer if implied).
If unknown, omit.
"""

def route_intent(question: str) -> dict:
    msg = INTENT_SYSTEM + "\nQuestion: " + question
    out = llm.invoke(msg).content.strip()
    # robust JSON parse:
    import json, re
    m = re.search(r"\{.*\}", out, re.S)
    if not m:
        return {"intent": "safety", "entities": {}}
    try:
        return json.loads(m.group(0))
    except Exception:
        return {"intent": "safety", "entities": {}}

In [ ]:
def latest_years(n=3) -> list[int]:
    dfy = run_sql("""
        SELECT crash_year
        FROM crashes
        WHERE crash_year IS NOT NULL
        GROUP BY crash_year
        ORDER BY crash_year DESC
        LIMIT :n
    """, {"n": n})
    return dfy["crash_year"].astype(int).tolist()

def normalize_borough(b):
    if not b: return None
    b = str(b).strip().upper()
    aliases = {
        "NYC": None, "NEW YORK": "MANHATTAN",
        "BK": "BROOKLYN", "QN": "QUEENS", "BX": "BRONX", "SI": "STATEN ISLAND"
    }
    return aliases.get(b, b)

def metric_columns(mode: str):
    mode = (mode or "cyclist").lower()
    if "ped" in mode:
        return ("number_of_pedestrians_injured", "number_of_pedestrians_killed")
    if "motor" in mode:
        return ("number_of_motorist_injured", "number_of_motorist_killed")
    if "cycl" in mode or "bike" in mode:
        return ("number_of_cyclist_injured", "number_of_cyclist_killed")
    return ("number_of_persons_injured", "number_of_persons_killed")

In [ ]:
import re

MONTHS = {
    "january": 1, "february": 2, "march": 3, "april": 4,
    "may": 5, "june": 6, "july": 7, "august": 8,
    "september": 9, "october": 10, "november": 11, "december": 12
}

def extract_year_month(question: str):
    q = question.lower()
    year = None
    month = None

    # year like 2024
    m = re.search(r"\b(20\d{2})\b", q)
    if m:
        year = int(m.group(1))

    # month name
    for name, num in MONTHS.items():
        if re.search(rf"\b{name}\b", q):
            month = num
            break

    return year, month

def extract_full_date(question: str):
    """
    Extract (year, month, day) from phrases like:
    - "January 1st 2024"
    - "On Jan 1"
    If year missing -> year=None
    """
    q = question.lower()

    year, month = extract_year_month(question)

    # day like 1st, 2nd, 3rd, 4th
    day = None
    dm = re.search(r"\b(\d{1,2})(st|nd|rd|th)?\b", q)
    if dm:
        d = int(dm.group(1))
        if 1 <= d <= 31:
            day = d

    return year, month, day

In [ ]:
def handle_identity(question: str):
    q = question.lower()
    if any(x in q for x in ["your name", "what is your name", "who are you"]):
        return one_liner(f"My name is {ASSISTANT_NAME}.", f"I’m {TAGLINE} powered by NYC crash open data")
    if any(x in q for x in ["what can you do", "capabilities", "help", "features"]):
        return one_liner(
            "I can answer crash counts by date/time, find hotspot intersections, summarize trends, and identify risky hours.",
            "I compute results from a SQLite crash database and summarize them briefly"
        )
    if any(x in q for x in ["how it works", "tech stack", "architecture", "built"]):
        return one_liner(
            "I use Python + SQLite for computation and a Groq-hosted Llama model for natural language routing and summaries.",
            "The database does the math so answers stay grounded in data"
        )
    return None

In [ ]:
import re

MONTHS = {
    "january": 1, "february": 2, "march": 3, "april": 4,
    "may": 5, "june": 6, "july": 7, "august": 8,
    "september": 9, "october": 10, "november": 11, "december": 12
}

def extract_year_month(question: str):
    q = question.lower()

    # year like 2025
    y = None
    m = re.search(r"\b(20\d{2})\b", q)
    if m:
        y = int(m.group(1))

    # month name like "January 2026"
    mo = None
    for name, num in MONTHS.items():
        if re.search(rf"\b{name}\b", q):
            mo = num
            break

    return y, mo

In [ ]:
def count_crashes(question: str, entities: dict):
    borough = normalize_borough(entities.get("borough"))
    year, month = extract_year_month(question)

    where = []
    params = {}

    if year:
        where.append("crash_year = :year")
        params["year"] = year
    if month:
        where.append("crash_month = :month")
        params["month"] = month
    if borough:
        where.append("borough = :borough")
        params["borough"] = borough

    if not where:
        # fallback: total crashes
        df = run_sql("SELECT COUNT(*) AS crashes FROM crashes;")
        c = int(df.iloc[0]["crashes"])
        return df, one_liner(f"{c} total crashes in the dataset.", "This is a direct COUNT(*) from the database")

    where_sql = "WHERE " + " AND ".join(where)
    df = run_sql(f"SELECT COUNT(*) AS crashes FROM crashes {where_sql};", params)
    c = int(df.iloc[0]["crashes"])

    parts = []
    if month and year:
        parts.append(f"{list(MONTHS.keys())[month-1].title()} {year}")
    elif year:
        parts.append(str(year))
    elif month:
        parts.append(f"{list(MONTHS.keys())[month-1].title()} (all years)")
    if borough:
        parts.append(borough.title())

    return df, one_liner(f"{c} crashes in " + ", ".join(parts) + ".", "This is a direct database count with your filters")

In [ ]:
def safety_hotspots(question: str, entities: dict, top_k: int = 10) -> tuple[pd.DataFrame, str]:
    borough = normalize_borough(entities.get("borough") or "BROOKLYN")
    mode = entities.get("mode") or "cyclist"
    involves_delivery = bool(entities.get("involves_delivery_truck", True))
    min_crashes = int(entities.get("min_crashes") or 10)

    injured_col, killed_col = metric_columns(mode)

    where = [
        "intersection IS NOT NULL",
        "TRIM(intersection) <> ''",
        "intersection LIKE '%@%'",

        # ✅ keep only intersections where BOTH sides of '@' are non-empty
        "TRIM(SUBSTR(intersection, 1, INSTR(intersection, '@') - 1)) <> ''",
        "TRIM(SUBSTR(intersection, INSTR(intersection, '@') + 1)) <> ''",
    ]
    params = {"min_crashes": min_crashes, "top_k": top_k}

    if borough:
        where.append("borough = :borough")
        params["borough"] = borough
    if involves_delivery:
        where.append("has_delivery_truck = 1")

    q = f"""
    SELECT
      intersection,
      COUNT(*) AS crashes,
      SUM(COALESCE({injured_col}, 0)) AS cyclist_injuries,
      1.0 * SUM(COALESCE({injured_col}, 0)) / COUNT(*) AS injury_rate
    FROM crashes
    WHERE {" AND ".join(where)}
    GROUP BY intersection
    HAVING COUNT(*) >= :min_crashes
    ORDER BY cyclist_injuries DESC, injury_rate DESC, crashes DESC
    LIMIT :top_k;
    """

    df = run_sql(q, params)

    if df.empty:
        return df, "No intersections matched the filters."

    # Answer-only format
    lines = [
        f"{i}. {r.intersection} — {int(r.crashes)} crashes, {int(r.cyclist_injuries)} cyclist injuries"
        for i, r in enumerate(df.itertuples(index=False), start=1)
    ]
    return df, "\n".join(lines)

In [ ]:
def policy_trend(question: str, entities: dict) -> tuple[pd.DataFrame, str]:
    borough = normalize_borough(entities.get("borough"))  # no default
    mode = entities.get("mode") or "persons"
    injured_col, killed_col = metric_columns(mode)

    # last N years default 3
    n = 3
    m = re.search(r"last\s+(\d+)\s+years", question.lower())
    if m:
        n = int(m.group(1))

    years = latest_years(n)
    params = {"y_latest": years[0], "y_earliest": years[-1]}
    where = ["crash_year BETWEEN :y_earliest AND :y_latest"]

    if borough:
        where.append("borough = :borough")
        params["borough"] = borough

    q = f"""
    SELECT
      crash_year,
      COUNT(*) AS crashes,
      SUM(COALESCE({injured_col}, 0)) AS injured,
      SUM(COALESCE({killed_col}, 0)) AS killed
    FROM crashes
    WHERE {" AND ".join(where)}
    GROUP BY crash_year
    ORDER BY crash_year ASC;
    """

    df = run_sql(q, params)

    # answer-only lines
    lines = [f"{int(r.crash_year)}: {int(r.crashes)} crashes" for r in df.itertuples(index=False)]
    return df, "\n".join(lines)

In [ ]:
def risk_by_time(question: str, entities: dict):
    borough = normalize_borough(entities.get("borough"))
    mode = entities.get("mode") or "persons"
    injured_col, killed_col = metric_columns(mode)

    where = []
    params = {}
    if borough:
        where.append("borough = :borough")
        params["borough"] = borough

    where_sql = ("WHERE " + " AND ".join(where)) if where else ""

    q = f"""
    SELECT
      crash_hour,
      COUNT(*) AS crashes,
      SUM(COALESCE({injured_col}, 0)) AS injured,
      1.0 * SUM(COALESCE({injured_col}, 0)) / COUNT(*) AS injuries_per_crash
    FROM crashes
    {where_sql}
    GROUP BY crash_hour
    HAVING COUNT(*) >= 5000
    ORDER BY injuries_per_crash DESC
    LIMIT 1;
    """

    df = run_sql(q, params)
    if df.empty:
        return df, one_liner("No result found.", "Not enough records to compute hourly risk")

    hr = int(df.iloc[0]["crash_hour"])
    ipr = float(df.iloc[0]["injuries_per_crash"])
    c = int(df.iloc[0]["crashes"])

    return df, one_liner(
        f"{hr}:00 is the riskiest hour (injuries/crash={ipr:.3f}, crashes={c}).",
        "This ranks hours by injuries-per-crash using historical crash records"
    )

In [ ]:
def peak_crash_year(question: str, entities: dict):
    borough = normalize_borough(entities.get("borough"))

    where = ["crash_year IS NOT NULL"]
    params = {}
    if borough:
        where.append("borough = :borough")
        params["borough"] = borough

    q = f"""
    SELECT crash_year, COUNT(*) AS crashes
    FROM crashes
    WHERE {" AND ".join(where)}
    GROUP BY crash_year
    ORDER BY crashes DESC
    LIMIT 1;
    """
    df = run_sql(q, params)
    y = int(df.iloc[0]["crash_year"])
    c = int(df.iloc[0]["crashes"])
    return df, one_liner(f"{y} ({c} crashes).", "This compares yearly totals across the full dataset")

In [ ]:
def accidents_on_date(question: str, entities: dict):
    year, month, day = extract_full_date(question)
    if not (month and day):
        return None

    borough = normalize_borough(entities.get("borough"))
    where = [
        "crash_month = :month",
        "CAST(strftime('%d', crash_datetime) AS INTEGER) = :day"
    ]
    params = {"month": month, "day": day}

    if year:
        where.append("crash_year = :year")
        params["year"] = year

    if borough:
        where.append("borough = :borough")
        params["borough"] = borough

    q = f"""
    SELECT COUNT(*) AS crashes
    FROM crashes
    WHERE {" AND ".join(where)};
    """
    df = run_sql(q, params)
    c = int(df.iloc[0]["crashes"])

    if year:
        label = f"{year:04d}-{month:02d}-{day:02d}"
    else:
        label = f"(any year) {month:02d}-{day:02d}"

    if c == 0:
        return df, one_liner(f"No. 0 crashes on {label}.", "This is counted from crash_datetime records")
    return df, one_liner(f"Yes. {c} crashes on {label}.", "This is counted from crash_datetime records")

In [ ]:
def extreme_accident_date(question: str, entities: dict):
    ql = question.lower()

    want_lowest = any(w in ql for w in ["lowest", "minimum", "min", "least", "smallest"])
    want_highest = any(w in ql for w in ["highest", "maximum", "max", "most", "peak", "largest"])
    order = "ASC" if (want_lowest and not want_highest) else "DESC"

    borough = normalize_borough(entities.get("borough"))
    year, month = extract_year_month(question)

    where = ["crash_date IS NOT NULL"]
    params = {}

    if borough:
        where.append("borough = :borough")
        params["borough"] = borough
    if year:
        where.append("crash_year = :year")
        params["year"] = year
    if month:
        where.append("crash_month = :month")
        params["month"] = month

    q = f"""
    SELECT crash_date, COUNT(*) AS crashes
    FROM crashes
    WHERE {" AND ".join(where)}
    GROUP BY crash_date
    ORDER BY crashes {order}, crash_date {order}
    LIMIT 1;
    """

    df = run_sql(q, params)
    if df.empty:
        return None

    d = df.iloc[0]["crash_date"]
    c = int(df.iloc[0]["crashes"])
    label = "lowest" if order == "ASC" else "highest"
    return df, one_liner(f"{d} ({c} crashes).", f"This is the {label} daily total based on crash_date counts")

In [ ]:
def peak_crash_hour(question: str, entities: dict):
    borough = normalize_borough(entities.get("borough"))

    where = []
    params = {}

    if borough:
        where.append("borough = :borough")
        params["borough"] = borough

    where_sql = ("WHERE " + " AND ".join(where)) if where else ""

    q = f"""
    SELECT crash_hour, COUNT(*) AS crashes
    FROM crashes
    {where_sql}
    GROUP BY crash_hour
    ORDER BY crashes DESC
    LIMIT 1;
    """

    df = run_sql(q, params)

    hr = int(df.iloc[0]["crash_hour"])
    c = int(df.iloc[0]["crashes"])

    return df, one_liner(
        f"{hr}:00 has the highest number of crashes ({c} crashes).",
        "This ranks hours by total crash count"
    )

In [ ]:
def safest_hour(question: str, entities: dict):
    borough = normalize_borough(entities.get("borough"))

    where = []
    params = {}

    if borough:
        where.append("borough = :borough")
        params["borough"] = borough

    where_sql = ("WHERE " + " AND ".join(where)) if where else ""

    q = f"""
    SELECT crash_hour, COUNT(*) AS crashes
    FROM crashes
    {where_sql}
    GROUP BY crash_hour
    HAVING COUNT(*) >= 5000
    ORDER BY crashes ASC
    LIMIT 1;
    """

    df = run_sql(q, params)

    hr = int(df.iloc[0]["crash_hour"])
    c = int(df.iloc[0]["crashes"])

    return df, one_liner(
        f"{hr}:00 is the safest hour ({c} crashes).",
        "This ranks hours by lowest crash frequency"
    )

In [ ]:
def safest_day_of_week(question: str, entities: dict):
    borough = normalize_borough(entities.get("borough"))

    where = []
    params = {}

    if borough:
        where.append("borough = :borough")
        params["borough"] = borough

    where_sql = ("WHERE " + " AND ".join(where)) if where else ""

    q = f"""
    SELECT day_of_week, COUNT(*) AS crashes
    FROM crashes
    {where_sql}
    GROUP BY day_of_week
    ORDER BY crashes ASC
    LIMIT 1;
    """

    df = run_sql(q, params)

    day = df.iloc[0]["day_of_week"]
    c = int(df.iloc[0]["crashes"])

    return df, one_liner(
        f"{day} is the safest day ({c} crashes).",
        "This ranks days by lowest crash frequency"
    )

In [ ]:
def most_crashes_day_of_week(question: str, entities: dict):
    q = """
    SELECT day_of_week, COUNT(*) AS crashes
    FROM crashes
    GROUP BY day_of_week
    ORDER BY crashes DESC
    LIMIT 1;
    """

    df = run_sql(q)

    day = df.iloc[0]["day_of_week"]
    c = int(df.iloc[0]["crashes"])

    return df, one_liner(
        f"{day} has the highest crash volume ({c} crashes).",
        "This ranks days by total crash frequency"
    )

In [ ]:
import datetime

def holiday_crashes(question: str, entities: dict):
    ql = question.lower()
    year, _ = extract_year_month(question)

    if "christmas" in ql:
        month = 12
        day = 25

    elif "thanksgiving" in ql:
        if not year:
            year = 2024  # fallback for demo
        # 4th Thursday of November
        d = datetime.date(year, 11, 1)
        thursdays = [d + datetime.timedelta(days=i)
                     for i in range(30)
                     if (d + datetime.timedelta(days=i)).month == 11
                     and (d + datetime.timedelta(days=i)).weekday() == 3]
        thanksgiving = thursdays[3]
        month = thanksgiving.month
        day = thanksgiving.day
    else:
        return None

    params = {"month": month, "day": day}
    where = [
        "crash_month = :month",
        "CAST(strftime('%d', crash_datetime) AS INTEGER) = :day"
    ]

    if year:
        where.append("crash_year = :year")
        params["year"] = year

    q = f"""
    SELECT COUNT(*) AS crashes
    FROM crashes
    WHERE {" AND ".join(where)};
    """

    df = run_sql(q, params)
    c = int(df.iloc[0]["crashes"])

    label = f"{month:02d}-{day:02d}"
    if year:
        label = f"{year}-{label}"

    return df, one_liner(
        f"{c} crashes recorded on {label}.",
        "This filters crash_datetime for the holiday date"
    )

In [ ]:
import re

def _extract_street_phrase(question: str) -> str:
    q = question.strip()

    # 1) If quoted, trust it
    m = re.search(r'"([^"]+)"', q) or re.search(r"'([^']+)'", q)
    if m:
        return m.group(1).strip()

    ql = q.lower()

    # 2) Common patterns: "crashes on X", "accidents on X", "how many ... on X"
    # Cut at time filters like "in 2024", "during January", "on Jan 1st"
    patterns = [
        r"(?:crashes|accidents)\s+on\s+(?P<street>.+)",
        r"how\s+many\s+(?:crashes|accidents).+?\s+on\s+(?P<street>.+)",
        r"(?:on|along)\s+(?P<street>.+)"
    ]

    for pat in patterns:
        m = re.search(pat, ql)
        if m:
            street = q[m.start("street"):].strip()  # slice from original string to preserve capitalization
            # remove trailing time phrases
            street = re.split(r"\s+\b(in|during|for|on)\b\s+", street, maxsplit=1)[0].strip()
            # remove trailing punctuation
            street = street.strip(" ?.,;:")
            return street

    # 3) If it's just a short input like "39 Street" use it as-is
    if len(q.split()) <= 5:
        return q.strip(' "')

    return q.strip(' "')

def crashes_by_street_name(question: str, entities: dict = None):
    """
    Counts crashes where a street appears in on/cross/off street fields or intersection.
    Supports year/month filters even if street isn't quoted.
    """
    entities = entities or {}

    street = _extract_street_phrase(question)
    year, month = extract_year_month(question)

    street_norm = street.upper()
    params = {"s": f"%{street_norm}%"}

    where = ["""
        (
            UPPER(COALESCE(on_street_name,'')) LIKE :s
         OR UPPER(COALESCE(cross_street_name,'')) LIKE :s
         OR UPPER(COALESCE(off_street_name,'')) LIKE :s
         OR UPPER(COALESCE(intersection,'')) LIKE :s
        )
    """]

    if year:
        where.append("crash_year = :year")
        params["year"] = year
    if month:
        where.append("crash_month = :month")
        params["month"] = month

    df = run_sql(f"""
        SELECT COUNT(*) AS crashes
        FROM crashes
        WHERE {" AND ".join(where)};
    """, params)

    c = int(df.iloc[0]["crashes"])

    label_parts = [street]
    if month and year:
        label_parts.append(f"{list(MONTHS.keys())[month-1].title()} {year}")
    elif year:
        label_parts.append(str(year))
    elif month:
        label_parts.append(f"{list(MONTHS.keys())[month-1].title()} (all years)")

    return df, one_liner(
        f"{c} crashes on {', '.join(label_parts)}.",
        "This counts crashes where the street appears in on/cross/off street or intersection fields"
    )

In [ ]:
def ask_city_ai(question: str):
    q = question.strip()
    ql = q.lower()

    # 0) Identity / capabilities
    out = handle_identity(q)
    if out:
        return out

    # LLM router used only after deterministic rules
    routed = route_intent(q)
    intent = routed.get("intent", "safety")
    entities = routed.get("entities", {}) or {}

    # 1) Holiday questions (Christmas / Thanksgiving)
    if ("christmas" in ql) or ("thanksgiving" in ql):
        out = holiday_crashes(q, entities)
        if out:
            _, ans = out
            return ans

    # 2) Highest / Lowest DATE questions
    if any(p in ql for p in [
        "which date",
        "highest number of accidents",
        "lowest number of accidents",
        "most accidents",
        "least accidents",
        "max accidents",
        "min accidents",
        "peak accidents"
    ]):
        out = extreme_accident_date(q, entities)
        if out:
            _, ans = out
            return ans

    # 3) Specific date questions (Jan 1st etc.)
    out = accidents_on_date(q, entities)
    if out:
        _, ans = out
        return ans

    # 4) Safest time/day (LOWEST crash volume)
    if any(p in ql for p in ["safest time to travel", "safest time", "safest hour"]):
        _, ans = safest_hour(q, entities)
        return ans

    if "safest day" in ql:
        _, ans = safest_day_of_week(q, entities)
        return ans

    # 5) Day of week with MOST crashes
    if ("day of the week" in ql or "day of week" in ql) and any(w in ql for w in ["most", "highest", "maximum", "max"]):
        _, ans = most_crashes_day_of_week(q, entities)
        return ans

    # Street-name quick query (if the input looks like a street question or just a street name)
    if any(x in ql for x in ["street", "st", "ave", "avenue", "blvd", "boulevard", "road", "rd", "parkway", "pkwy", "drive", "dr"]) \
       or (len(q.split()) <= 4 and not any(x in ql for x in ["how many", "which date", "highest", "lowest", "safest", "riskiest", "year", "month", "day of week"])):
        # Example: "Ocean Parkway" or "How many crashes on Ocean Parkway?"
        if "crash" in ql or "accident" in ql or len(q.split()) <= 4:
            _, ans = crashes_by_street_name(q, entities)
            return ans
    # Street-name queries (supports year/month like "Atlantic Ave in 2024")
    if ("crash" in ql or "accident" in ql) and any(x in ql for x in ["ave", "avenue", "street", "st", "blvd", "boulevard", "road", "rd", "parkway", "pkwy", "drive", "dr"]):
        _, ans = crashes_by_street_name(q, entities)
        return ans
    # 6) Count questions (month/year filters)
    wants_count = any(p in ql for p in [
        "how many",
        "number of crashes",
        "count of crashes",
        "total crashes",
        "any accidents"
    ])
    year, month = extract_year_month(q)

    if wants_count and (year or month):
        _, ans = count_crashes(q, entities)
        return ans

    # 7) Peak Year (across ALL years)
    if any(w in ql for w in ["highest", "most", "maximum", "max", "peak"]) and ("year" in ql) and ("crash" in ql or "accident" in ql):
        _, ans = peak_crash_year(q, entities)
        return ans

    # 8) Peak Crash Hour (VOLUME metric)
    if ("highest number of crashes" in ql or "most crashes" in ql) and ("hour" in ql or "time of day" in ql):
        _, ans = peak_crash_hour(q, entities)
        return ans

    # 9) Risk / Severity Hour (SEVERITY metric)
    is_volume_question = ("highest number of crashes" in ql) or ("most crashes" in ql)

    if (not is_volume_question) and (
        intent == "risk" or
        any(w in ql for w in ["risky", "riskiest", "most dangerous", "safest time"])
    ):
        _, ans = risk_by_time(q, entities)
        return ans

    # 10) Policy Trend
    if intent == "policy":
        _, ans = policy_trend(q, entities)
        return one_liner(ans, "This summarizes trends using computed yearly totals")

    # 11) Default → Safety Hotspots
    _, ans = safety_hotspots(q, entities, top_k=10)
    return one_liner(ans, "This ranks intersections by crash volume and injury totals")

# 👋 Hello I'm VisionZeroMate your NYC Crash & Safety Assistant

In [ ]:
print(ask_city_ai("What is your name?"))

My name is VisionZeroMate.
I’m NYC Crash & Safety Assistant powered by NYC crash open data.


In [ ]:
print(ask_city_ai("How many crashes on 1st July 2012?"))

Yes. 538 crashes on 2012-07-01.
This is counted from crash_datetime records.


In [ ]:
print(ask_city_ai("On thanksgiving 2023 were there any accidents?"))

197 crashes recorded on 2023-11-23.
This filters crash_datetime for the holiday date.


In [ ]:
print(ask_city_ai("On which date there were highest number of accidents?"))

2014-01-21 (1161 crashes).
This is the highest daily total based on crash_date counts.


In [ ]:
print(ask_city_ai("On which date there were lowest number of accidents?"))

2020-04-05 (94 crashes).
This is the lowest daily total based on crash_date counts.


In [ ]:
print(ask_city_ai("Which intersections in queens have the highest cyclist injuries involving delivery trucks?"))

1. 39 STREET @ 43 AVENUE — 13 crashes, 3 cyclist injuries
2. GRAND AVENUE @ 49 PLACE — 10 crashes, 2 cyclist injuries
3. VANDAM STREET @ REVIEW AVENUE — 13 crashes, 2 cyclist injuries
4. 55 DRIVE @ MAURICE AVENUE — 10 crashes, 1 cyclist injuries
5. 69 STREET @ ROOSEVELT AVENUE — 10 crashes, 1 cyclist injuries
6. FRESH POND ROAD @ MADISON STREET — 10 crashes, 1 cyclist injuries
7. QUEENS BOULEVARD @ 38 STREET — 10 crashes, 1 cyclist injuries
8. QUEENS BOULEVARD @ 58 STREET — 10 crashes, 1 cyclist injuries
9. SKILLMAN AVENUE @ QUEENS BOULEVARD — 10 crashes, 1 cyclist injuries
10. 108 STREET @ NORTHERN BOULEVARD — 11 crashes, 1 cyclist injuries
This ranks intersections by crash volume and injury totals.


In [ ]:
print(ask_city_ai("What time of day is riskiest for injuries citywide?"))

23:00 is the riskiest hour (injuries/crash=0.425, crashes=62816).
This ranks hours by injuries-per-crash using historical crash records.


In [ ]:
print(ask_city_ai("Which is the safest time to travel?"))

3:00 is the safest hour (27177 crashes).
This ranks hours by lowest crash frequency.


In [ ]:
print(ask_city_ai("Which day of the week has most crashes recorded?"))

Friday has the highest crash volume (357076 crashes).
This ranks days by total crash frequency.


In [ ]:
print(ask_city_ai("How many crashes on Christmas 2024?"))

149 crashes recorded on 2024-12-25.
This filters crash_datetime for the holiday date.


In [ ]:
print(ask_city_ai('How many accidents on 39 Street?'))

9230 crashes on 39 Street.
This counts crashes where the street appears in on/cross/off street or intersection fields.


In [ ]:
print(ask_city_ai("Crashes on 39 Street in 2024"))

260 crashes on 39 Street, 2024.
This counts crashes where the street appears in on/cross/off street or intersection fields.
